In [2]:
# שלב 1: ייבוא ספריות, קביעת random_state=42 (דרישת המטלה) ובדיקת גרסאות
import pandas as pd
import numpy as np
import sklearn, sys

RANDOM_STATE = 42  # Required by the assignment: random_state=42 everywhere
print(sys.version)
print("pandas", pd.__version__, "| sklearn", sklearn.__version__)

3.12.16 | packaged by Anaconda, Inc. | (main, Oct 26 2026, 20:13:05) [MSC v.1942 64 bit (AMD64)]
pandas 2.2.2 | sklearn 1.5.1


In [3]:
# שלב 2.1: טעינת הדאטה והצגת 5 שורות אקראיות (דגימה מייצגת יותר מהשורות הראשונות)
df = pd.read_csv("dataset.csv")
print(df.shape)
df.sample(5, random_state=RANDOM_STATE)

(45000, 30)


,customer_id_hash,agent_code,country,area_type,region_density,winter_severity,driver_age,gender,marital_status,license_years,...,annual_km_estimated,pct_intercity,pct_night_driving,parking_type,current_premium,claim_count,at_fault,days_to_settlement,police_report_filed,annual_claim_cost
37979,0ebf89f0,A020,FR,suburban,806,3.0,63,M,married,36,...,66500.0,10,45,garage,4270,0,NaN,NaN,NaN,0
9911,e7bedf11,A143,US,rural,24,0.0,38,M,divorced,20,...,17500.0,15,5,garage,1860,0,NaN,NaN,NaN,0
43386,9ff342a8,A060,FR,suburban,1115,3.8,84,M,married,66,...,3000.0,0,10,garage,1950,0,NaN,NaN,NaN,0
13822,b66c36b7,A004,DE,urban,10949,6.6,48,M,divorced,30,...,NaN,20,30,lot,13810,0,NaN,NaN,NaN,0
44810,30fbec1d,A004,US,rural,100,6.4,80,M,married,62,...,15500.0,55,10,garage,5160,0,NaN,NaN,NaN,0


In [4]:
# שלב 2.2: סוג הנתונים בכל עמודה, כדי לזהות עמודות מספריות שנקראו כטקסט
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 30 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   customer_id_hash       45000 non-null  object 
 1   agent_code             45000 non-null  object 
 2   country                45000 non-null  object 
 3   area_type              45000 non-null  object 
 4   region_density         45000 non-null  int64  
 5   winter_severity        45000 non-null  float64
 6   driver_age             45000 non-null  int64  
 7   gender                 45000 non-null  object 
 8   marital_status         45000 non-null  object 
 9   license_years          45000 non-null  int64  
 10  prior_claims_5y        41858 non-null  float64
 11  no_claims_years        45000 non-null  int64  
 12  violations_3y          45000 non-null  int64  
 13  vehicle_type           45000 non-null  object 
 14  vehicle_age            45000 non-null  int64  
 15  en

In [5]:
# שלב 2.3: כמות הערכים החסרים בכל עמודה (רק עמודות שיש בהן חוסרים)
missing = df.isna().sum()
missing[missing > 0]

prior_claims_5y         3142
annual_km_estimated     4311
at_fault               29207
days_to_settlement     29207
police_report_filed    29207
dtype: int64

In [6]:
# שלב 2.4: סטטיסטיקה של העמודות המספריות, כדי לזהות ערכים חריגים ולא הגיוניים (min/max)
df.describe().T

,count,mean,std,min,25%,50%,75%,max
region_density,45000.0,1886.716311,2161.133202,4.0,498.0,1151.0,2593.0,20000.0
winter_severity,45000.0,3.459389,2.214855,0.0,1.7,3.3,5.0,10.0
driver_age,45000.0,48.010511,16.613355,18.0,35.0,47.0,60.0,90.0
license_years,45000.0,28.853867,16.866459,0.0,15.0,28.0,41.0,95.0
prior_claims_5y,41858.0,1.039706,1.384058,0.0,0.0,1.0,2.0,6.0
no_claims_years,45000.0,6.750933,8.101940,0.0,1.0,3.0,10.0,30.0
violations_3y,45000.0,0.416467,0.717039,0.0,0.0,0.0,1.0,6.0
vehicle_age,45000.0,7.638844,5.313188,0.0,4.0,6.0,10.0,25.0
engine_hp,45000.0,143.920800,66.350986,12.0,94.0,137.0,178.0,650.0
safety_features_count,45000.0,3.983378,2.028846,0.0,2.0,4.0,6.0,8.0


## 3. זיהוי Data Leakage
המודל חוזה עלות תביעות של נהג חדש בתחילת השנה. מותר להשתמש רק בשדות שידועים לפני חתימת החוזה.
בשלב זה נבדוק כל עמודה חשודה ונוכיח בנתונים אם היא מדליפה מידע מהעתיד.

In [9]:
# בדיקה האם מספר התביעות חופף בדיוק לעלות התביעות
pd.crosstab(df["claim_count"] > 0, df["annual_claim_cost"] > 0,
            rownames=["claim_count > 0"], colnames=["cost > 0"])

cost > 0,False,True
claim_count > 0,,
False,29207,0
True,0,15793


In [10]:
# בדיקה האם שלוש עמודות התאונה ריקות בדיוק כשאין תביעה
for col in ["at_fault", "days_to_settlement", "police_report_filed"]:
    print(col)
    print(pd.crosstab(df[col].isna(), df["annual_claim_cost"] > 0,
                      rownames=["missing"], colnames=["cost > 0"]), "\n")

at_fault
cost > 0  False  True 
missing               
False         0  15793
True      29207      0 

days_to_settlement
cost > 0  False  True 
missing               
False         0  15793
True      29207      0 

police_report_filed
cost > 0  False  True 
missing               
False         0  15793
True      29207      0 



In [11]:
# בדיקת הקשר בין הפרמיה של סוף השנה לבין עלות התביעות
print("Correlation with target:", round(df["current_premium"].corr(df["annual_claim_cost"]), 3))
df.groupby(df["annual_claim_cost"] > 0)["current_premium"].mean()

Correlation with target: 0.577


annual_claim_cost
False    3188.265484
True     6214.693852
Name: current_premium, dtype: float64

In [12]:
# בדיקה האם שנות הנהיגה ללא תביעה עודכנו בסוף השנה
df.groupby(df["at_fault"].fillna(-1))["no_claims_years"].apply(lambda s: (s == 0).mean())

at_fault
-1.0    0.146506
 0.0    0.264279
 1.0    0.347816
Name: no_claims_years, dtype: float64

In [13]:
# בדיקת העמודות המנהליות מזהה לקוח וקוד סוכן
print("customer_id_hash unique:", df["customer_id_hash"].is_unique)
print("agent_code values:", df["agent_code"].nunique())

customer_id_hash unique: True
agent_code values: 300


### מסקנות שלב 3
עמודות Leakage שיוסרו: claim_count, at_fault, days_to_settlement, police_report_filed, current_premium. כולן נוצרות במהלך שנת הביטוח או בסופה ולכן אינן ידועות ביום החתימה.
עמודה שתוסר כי אין לה ערך לחיזוי: customer_id_hash.
עמודה שנבדקה ונמצאה תקינה: no_claims_years, כי רק 35% מהאשמים בתאונה השנה מופיעים עם 0, ולכן השדה לא עודכן בסוף השנה.
החלטה פתוחה: agent_code.

In [14]:
# רשימת העמודות שיוסרו מהמודל בגלל דליפת מידע או חוסר ערך
LEAKAGE_COLS = ["claim_count", "at_fault", "days_to_settlement",
                "police_report_filed", "current_premium"]
ID_COLS = ["customer_id_hash"]